# STEP 4A: Mask-to-Feature Extraction, Validation Correlation & Human Labeling Sheet
**Road PCI Intelligence**

This notebook executes on **CPU** to:
1. Compute geometric and morphological distress features from **Ground-Truth** masks across train+val splits ($N = 1,486$ images).
2. Export `manifests/features_gt_trainval.csv` and report feature distributions (median, 10th percentile, 90th percentile).
3. Render visual inspection figures showing 6 validation samples per source (`image | mask | skeleton with junctions marked`).
4. Evaluate feature fidelity on validation predictions using the frozen model (`best.pt`, Naive Multi-Task Loss, Flip-TTA, threshold 0.5) via **Spearman rank correlation**.
5. Generate the blinded human labeling sheet (60 test pairs across 10 pages, with `labels/pairs_template.csv`).

In [ ]:
# ==============================================================================
# 1. ENVIRONMENT SETUP & DEPENDENCIES
# ==============================================================================
import os
import sys
import subprocess
import shutil
import time
from pathlib import Path

print("Installing required dependencies (segmentation-models-pytorch, albumentations, scipy)...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "segmentation-models-pytorch==0.5.0", "albumentations==2.0.8", "scipy==1.17.1"], check=True)

# Clone repository
repo_dir = Path("/kaggle/working/repo")
if repo_dir.exists():
    shutil.rmtree(repo_dir)

print("Cloning repository from GitHub...")
subprocess.run(["git", "clone", "https://github.com/skilloutofbounds/road-pci-intelligence.git", str(repo_dir)], check=True)
sys.path.insert(0, str(repo_dir))

import torch
import torchvision
import numpy as np
import pandas as pd
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
import segmentation_models_pytorch as smp

print("=" * 80)
print("ENVIRONMENT INFORMATION:")
print(f"  • Python:       {sys.version.split()[0]}")
print(f"  • PyTorch:      {torch.__version__}")
print(f"  • Torchvision:  {torchvision.__version__}")
print(f"  • SMP Version:  {smp.__version__}")
print(f"  • CPU Threads:  {torch.get_num_threads()}")
print("=" * 80)

In [ ]:
# ==============================================================================
# 2. RESOLVE DATASET ROOTS & VERIFY FINAL WEIGHTS
# ==============================================================================
import hashlib
from src.build_manifest import DATASET_SPECS, resolve_dataset_root
from src.pavement_features import compute_features, rescale_mask, compute_skeleton_length
from src.data_rules import load_standardized_mask
from src.datasets import RoadDistressDataset

# Locate datasets
dataset_roots = {}
for src in ['crack500', 'road_crack', 'pothole']:
    meta = DATASET_SPECS[src]
    resolved = resolve_dataset_root(meta['owner'], meta['name'])
    if resolved is None:
        raise FileNotFoundError(f"CRITICAL: Failed to locate dataset root for {src}")
    dataset_roots[src] = resolved
    print(f"  • {src:<12} -> {resolved}")

# Verify weights
weights_candidates = [
    Path("/kaggle/input/datasets/vedshah04/road-pci-seg-final-weights/best.pt"),
    Path("/kaggle/input/road-pci-seg-final-weights/best.pt"),
    repo_dir / "scratch/downloaded_final_weights/best.pt"
]
weights_path = None
for p in weights_candidates:
    if p.exists():
        weights_path = p
        break

if weights_path is None:
    raise FileNotFoundError("CRITICAL: Failed to locate best.pt final weights checkpoint!")

print(f"Found weights at: {weights_path} ({weights_path.stat().st_size:,} bytes)")
sha256 = hashlib.sha256(weights_path.read_bytes()).hexdigest()
print(f"Computed SHA-256: {sha256}")
EXPECTED_SHA = "9269c5edb6e5d37b359bd3337fbd83c60901cd2b241f9ef789e9116fee400ef7"
assert sha256 == EXPECTED_SHA, f"SHA mismatch! Expected {EXPECTED_SHA}, got {sha256}"
print("[PASS] SHA-256 verified strictly.")

In [ ]:
# ==============================================================================
# 3. PHASE 1: COMPUTE GROUND-TRUTH FEATURES ON TRAIN + VAL (N = 1,486)
# ==============================================================================
manifest_path = repo_dir / 'manifests/segmentation_manifest.csv'
manifest_df = pd.read_csv(manifest_path)

trainval_df = manifest_df[manifest_df['split'].isin(['train', 'val'])].copy().reset_index(drop=True)
print(f"Loaded train+val records: {len(trainval_df):,} images")
print(f"  • crack500:   {(trainval_df['source'] == 'crack500').sum():,} images")
print(f"  • road_crack: {(trainval_df['source'] == 'road_crack').sum():,} images")
print(f"  • pothole:    {(trainval_df['source'] == 'pothole').sum():,} images")

t0 = time.time()
gt_feature_records = []

for idx, row in trainval_df.iterrows():
    src = row['source']
    split = row['split']
    root = dataset_roots[src]
    mask_rel = Path(row['mask_path_or_label_path'])
    full_mask_path = root / mask_rel
    img_shape = (int(row['height']), int(row['width']))
    
    # Load standardized binary mask
    raw_mask = load_standardized_mask(source=src, mask_or_label_path=full_mask_path, img_shape=img_shape)
    
    if src in ['crack500', 'road_crack']:
        crack_m = raw_mask
        pothole_m = None
    elif src == 'pothole':
        crack_m = None
        pothole_m = raw_mask
    else:
        crack_m = None
        pothole_m = None
        
    feats = compute_features(crack_mask=crack_m, pothole_mask=pothole_m)
    
    rec = {
        'source': src,
        'image_path': row['image_path'],
        'split': split,
        'group_id': row['group_id'],
        **feats
    }
    gt_feature_records.append(rec)
    
    if (idx + 1) % 400 == 0 or (idx + 1) == len(trainval_df):
        print(f"  Processed {idx + 1:,} / {len(trainval_df):,} masks ({((idx+1)/len(trainval_df))*100:.1f}%) in {time.time()-t0:.1f}s")

gt_features_df = pd.DataFrame(gt_feature_records)
print(f"[OK] Extracted features for {len(gt_features_df):,} ground-truth masks in {time.time()-t0:.2f}s")

# Save to CSV
out_csv = Path("/kaggle/working/features_gt_trainval.csv")
gt_features_df.to_csv(out_csv, index=False)
print(f"[OK] Exported features to {out_csv}")

# Also copy into repo manifests
manifests_dest = repo_dir / "manifests/features_gt_trainval.csv"
gt_features_df.to_csv(manifests_dest, index=False)
print(f"[OK] Copied features to {manifests_dest}")

In [ ]:
# ==============================================================================
# 4. PRINT FEATURE DISTRIBUTIONS: MEDIAN, P10, P90 PER SOURCE
# ==============================================================================
feature_cols = [
    'crack_area_frac',
    'crack_length_px',
    'crack_length_density',
    'mean_width_px',
    'relative_width',
    'max_width_px',
    'width_p90_px',
    'n_crack_components',
    'n_junctions',
    'cells',
    'network_index',
    'pothole_count',
    'pothole_area_frac',
    'largest_pothole_frac'
]

print("=" * 105)
print(f"{'Source':<12} | {'Feature':<22} | {'Median':<14} | {'10th Pct (P10)':<16} | {'90th Pct (P90)':<16}")
print("=" * 105)

for src in ['crack500', 'road_crack', 'pothole']:
    src_df = gt_features_df[gt_features_df['source'] == src]
    print(f"--- SOURCE: {src.upper()} (N = {len(src_df)}) ---")
    for feat in feature_cols:
        vals = src_df[feat].to_numpy()
        med = np.median(vals)
        p10 = np.percentile(vals, 10)
        p90 = np.percentile(vals, 90)
        
        if feat in ['n_crack_components', 'n_junctions', 'cells', 'pothole_count']:
            print(f"{src:<12} | {feat:<22} | {med:14.1f} | {p10:16.1f} | {p90:16.1f}")
        elif 'frac' in feat or 'relative' in feat:
            print(f"{src:<12} | {feat:<22} | {med:14.6f} | {p10:16.6f} | {p90:16.6f}")
        else:
            print(f"{src:<12} | {feat:<22} | {med:14.3f} | {p10:16.3f} | {p90:16.3f}")
    print("-" * 105)

In [ ]:
# ==============================================================================
# 5. VISUAL INSPECTION: 6 VAL SAMPLES PER SOURCE (IMAGE | MASK | SKELETON + JUNCTIONS)
# ==============================================================================
from skimage.morphology import skeletonize
from src.pavement_features import compute_skeleton_length

val_df = trainval_df[trainval_df['split'] == 'val'].reset_index(drop=True)

for src in ['crack500', 'road_crack', 'pothole']:
    src_val = val_df[val_df['source'] == src].reset_index(drop=True)
    # Pick 6 diverse validation samples
    n_total = len(src_val)
    step = max(1, n_total // 6)
    indices = [min(i * step, n_total - 1) for i in range(6)]
    indices = sorted(list(dict.fromkeys(indices)))
    while len(indices) < 6:
        indices.append(len(indices))
        
    samples = src_val.iloc[indices]
    
    fig, axes = plt.subplots(6, 3, figsize=(15, 24))
    plt.subplots_adjust(top=0.96, bottom=0.03, hspace=0.32, wspace=0.10)
    fig.suptitle(f"Ground-Truth Feature Inspection: {src.upper()} Validation Set (6 Samples)", fontsize=14, fontweight='bold', y=0.98)
    
    for row_idx, (_, row) in enumerate(samples.iterrows()):
        root = dataset_roots[src]
        img_full = root / row['image_path']
        mask_full = root / row['mask_path_or_label_path']
        img_shape = (int(row['height']), int(row['width']))
        
        with Image.open(img_full) as pil_img:
            img_rgb = np.array(pil_img.convert('RGB'))
            
        raw_mask = load_standardized_mask(src, mask_full, img_shape=img_shape)
        
        # Rescale image and mask to 512 shorter side
        mask_512 = rescale_mask(raw_mask, target_shorter_side=512)
        h5, w5 = mask_512.shape[:2]
        img_512 = cv2.resize(img_rgb, (w5, h5), interpolation=cv2.INTER_LINEAR)
        
        stem = Path(row['image_path']).name
        if len(stem) > 24:
            stem = stem[:21] + "..."
            
        # Col 0: Image
        axes[row_idx, 0].imshow(img_512)
        axes[row_idx, 0].set_title(f"[Sample {row_idx+1}] {stem}\nImage ({w5}x{h5})", fontsize=8.5, fontweight='bold')
        axes[row_idx, 0].axis('off')
        
        # Col 1: Mask Overlay
        mask_overlay = img_512.copy()
        color = [230, 30, 30] if src != 'pothole' else [30, 110, 240]
        fg = (mask_512 > 0)
        mask_overlay[fg] = (mask_overlay[fg].astype(float) * 0.45 + np.array(color) * 0.55).astype(np.uint8)
        
        axes[row_idx, 1].imshow(mask_overlay)
        axes[row_idx, 1].set_title(f"GT Mask ({'crack' if src != 'pothole' else 'pothole'})\nPixels: {int(fg.sum()):,} ({fg.mean()*100:.2f}%)", fontsize=8.5)
        axes[row_idx, 1].axis('off')
        
        # Col 2: Skeleton with Junctions
        skel_canvas = img_512.copy()
        if src != 'pothole':
            # Crack cleaning
            n_lbl, lbls, stats, _ = cv2.connectedComponentsWithStats(mask_512, connectivity=8)
            clean_crack = np.zeros_like(mask_512)
            for l in range(1, n_lbl):
                if stats[l, cv2.CC_STAT_AREA] >= 30:
                    clean_crack[lbls == l] = 1
            skel = skeletonize(clean_crack > 0)
            
            # Skeleton in bright green
            skel_canvas[skel] = [0, 255, 0]
            
            # Junctions
            sk_u8 = skel.astype(np.uint8)
            k_n = np.array([[1, 1, 1], [1, 0, 1], [1, 1, 1]], dtype=np.uint8)
            nc = cv2.filter2D(sk_u8, -1, k_n) * sk_u8
            junc_pts = ((nc >= 3) & (sk_u8 > 0))
            
            jy, jx = np.where(junc_pts)
            for y_j, x_j in zip(jy, jx):
                cv2.circle(skel_canvas, (x_j, y_j), 4, (255, 0, 0), thickness=-1)
                
            length_val = compute_skeleton_length(skel)
            num_j_lbl, _ = cv2.connectedComponents(junc_pts.astype(np.uint8), connectivity=8)
            n_j = max(0, num_j_lbl - 1)
            axes[row_idx, 2].imshow(skel_canvas)
            axes[row_idx, 2].set_title(f"Skeleton (Green) & Junctions (Red)\nLength: {length_val:.1f} px | Junctions: {n_j}", fontsize=8.5, fontweight='bold')
        else:
            # Pothole: show component contours
            n_lbl, lbls, stats, _ = cv2.connectedComponentsWithStats(mask_512, connectivity=8)
            p_cnt = 0
            for l in range(1, n_lbl):
                if stats[l, cv2.CC_STAT_AREA] >= 50:
                    p_cnt += 1
                    comp_mask = (lbls == l).astype(np.uint8)
                    contours, _ = cv2.findContours(comp_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                    cv2.drawContours(skel_canvas, contours, -1, (255, 255, 0), 2)
            axes[row_idx, 2].imshow(skel_canvas)
            axes[row_idx, 2].set_title(f"Pothole Contours (Yellow)\nValid Potholes (>= 50 px): {p_cnt}", fontsize=8.5, fontweight='bold')
        axes[row_idx, 2].axis('off')
        
    out_fig = Path(f"/kaggle/working/val_feature_skeletons_{src}.png")
    plt.savefig(out_fig, dpi=120, bbox_inches='tight')
    plt.close()
    print(f"[OK] Saved figure: {out_fig}")

In [ ]:
# ==============================================================================
# 6. PHASE 3: VALIDATION PREDICTED MASKS & SPEARMAN RANK CORRELATION
# ==============================================================================
# Load frozen model
device = torch.device('cpu')
model = smp.Unet(encoder_name='resnet34', classes=2)
state_dict = torch.load(weights_path, map_location=device)
model.load_state_dict(state_dict)
model.eval()
print(f"[OK] Pretrained ResNet-34 U-Net loaded on CPU ({sum(p.numel() for p in model.parameters()):,} parameters)")

val_dataset = RoadDistressDataset(manifest_df=manifest_df, dataset_roots=dataset_roots, split='val', crop_size=512)
print(f"Validation DataLoader: {len(val_dataset)} images")

t0_eval = time.time()
paired_val_records = []

with torch.no_grad():
    for idx in range(len(val_dataset)):
        image_t, target_t, label_mask, metas = val_dataset[idx]
        image_b = image_t.unsqueeze(0) # [1, 3, H, W]
        src = metas['source']
        unpad_info = metas['eval_unpad']
        y0, x0, h, w = unpad_info
        
        # 1. Forward Flip-TTA on CPU
        logits_orig = model(image_b)
        probs_orig = torch.sigmoid(logits_orig)
        
        img_h = torch.flip(image_b, dims=[-1])
        logits_h = model(img_h)
        probs_h = torch.flip(torch.sigmoid(logits_h), dims=[-1])
        
        img_v = torch.flip(image_b, dims=[-2])
        logits_v = model(img_v)
        probs_v = torch.flip(torch.sigmoid(logits_v), dims=[-2])
        
        probs_tta = (probs_orig + probs_h + probs_v) / 3.0
        
        # Unpad
        p_crack = (probs_tta[0, 0, y0:y0+h, x0:x0+w] >= 0.5).numpy().astype(np.uint8)
        p_pot = (probs_tta[0, 1, y0:y0+h, x0:x0+w] >= 0.5).numpy().astype(np.uint8)
        
        # GT unpad
        t_crack = (target_t[0, y0:y0+h, x0:x0+w] >= 0.5).numpy().astype(np.uint8) if label_mask[0] else None
        t_pot = (target_t[1, y0:y0+h, x0:x0+w] >= 0.5).numpy().astype(np.uint8) if label_mask[1] else None
        
        # Compute features
        gt_feats = compute_features(t_crack, t_pot)
        pred_feats = compute_features(p_crack, p_pot)
        
        paired_val_records.append({
            'source': src,
            'image_path': metas['image_path'],
            'gt': gt_feats,
            'pred': pred_feats
        })
        
        if (idx + 1) % 50 == 0 or (idx + 1) == len(val_dataset):
            print(f"  Evaluated {idx + 1} / {len(val_dataset)} images ({((idx+1)/len(val_dataset))*100:.1f}%) in {time.time()-t0_eval:.1f}s")

print(f"[OK] Validation inference and feature extraction completed in {time.time()-t0_eval:.2f}s")

# Compute Spearman Correlation per source
eval_targets = {
    'crack500': ['crack_length_px', 'relative_width', 'network_index'],
    'road_crack': ['crack_length_px', 'relative_width', 'network_index'],
    'pothole': ['pothole_area_frac']
}

print("\n" + "=" * 95)
print("SPEARMAN RANK CORRELATION: GROUND-TRUTH vs. PREDICTED FEATURES (VALIDATION SPLIT)")
print("=" * 95)
print(f"{'Source':<12} | {'N Images':<10} | {'Feature':<22} | {'Spearman rho':<16} | {'p-value':<14} | {'Significance':<12}")
print("-" * 95)

correlation_results = {}
for src, feat_list in eval_targets.items():
    src_recs = [r for r in paired_val_records if r['source'] == src]
    n_img = len(src_recs)
    correlation_results[src] = {}
    
    for feat in feat_list:
        gt_vals = np.array([r['gt'][feat] for r in src_recs])
        pred_vals = np.array([r['pred'][feat] for r in src_recs])
        
        # Check if constant
        if np.std(gt_vals) == 0 or np.std(pred_vals) == 0:
            rho, pval = float('nan'), float('nan')
            sig = "Constant"
        else:
            res = spearmanr(gt_vals, pred_vals)
            rho, pval = float(res.statistic), float(res.pvalue)
            sig = "*** (p<0.001)" if pval < 0.001 else "** (p<0.01)" if pval < 0.01 else "* (p<0.05)" if pval < 0.05 else "n.s."
            
        correlation_results[src][feat] = {'n': n_img, 'rho': rho, 'pval': pval}
        print(f"{src:<12} | {n_img:<10} | {feat:<22} | {rho:16.4f} | {pval:14.2e} | {sig:<12}")
    print("-" * 95)
print("=" * 95)

In [ ]:
# ==============================================================================
# 7. PHASE 4: GENERATE HUMAN LABELING SHEET (TEST SPLIT ONLY, 60 PAIRS)
# ==============================================================================
import subprocess

print("Running scripts/make_pair_sheet.py...")
out_labels_dir = Path("/kaggle/working/labels")
out_labels_dir.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable,
    str(repo_dir / "scripts/make_pair_sheet.py"),
    "--manifest", str(repo_dir / "manifests/segmentation_manifest.csv"),
    "--out-csv", str(out_labels_dir / "pairs_template.csv"),
    "--out-dir", str(out_labels_dir),
    "--seed", "42"
]
res = subprocess.run(cmd, capture_output=True, text=True)
print("STDOUT:", res.stdout)
if res.stderr:
    print("STDERR:", res.stderr)

# Copy template CSV to repo
shutil.copy2(out_labels_dir / "pairs_template.csv", repo_dir / "labels/pairs_template.csv")
print("[OK] Verified labels/pairs_template.csv in repo directory.")

In [ ]:
# ==============================================================================
# 8. EXPORT FINAL STEP 4A SUMMARY JSON
# ==============================================================================
import json

summary_payload = {
    'weights_sha256': sha256,
    'trainval_records': len(gt_features_df),
    'val_correlation': correlation_results,
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')
}

with open("/kaggle/working/step_4a_summary.json", "w") as f:
    json.dump(summary_payload, f, indent=2)
print("[OK] Exported /kaggle/working/step_4a_summary.json successfully.")